In [2]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# The file paths
files = {
    "ResNet-34": "outputs/resnet34/resnet34_multirun_per_run.csv",
    "ResNet-50": "outputs/resnet50/resnet50_multirun_per_run.csv",
    "DenseNet-121": "outputs/densenet121/densenet121_multirun_per_run.csv",
    "EfficientNet-B0": "outputs/efficientnetb0/efficientnetb0_multirun_per_run.csv",
    "Inception-V3": "outputs/inceptionv3/inceptionv3_multirun_per_run.csv",
    "VGG-16": "outputs/vgg16/vgg16_multirun_per_run.csv",
}

# Load and combine all model results
all_dfs = []

for model_name, file_path in files.items():
    df = pd.read_csv(file_path)
    df["model"] = model_name
    all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

print("Combined shape:", combined_df.shape)
print("\nColumns found:")
print(combined_df.columns.tolist())
print("\nModels:")
print(combined_df["model"].value_counts())


# Check the data
print("\nFirst few rows:")
print(combined_df.head())

print("\nUnique seeds:")
if "seed" in combined_df.columns:
    print(sorted(combined_df["seed"].unique()))


# Select internal test with TTA
test_df = combined_df.copy()
print("\nRows used for statistical analysis:")
print(test_df.shape)

# Define metrics to analyse
metrics = [
    "internal_tta.accuracy",
    "internal_tta.precision_macro",
    "internal_tta.recall_macro",
    "internal_tta.f1_macro",
    "internal_tta.macro_roc_auc"
]
models = list(files.keys())


# ============================================================
# 6. CHECK THAT ALL REQUIRED METRICS EXIST
# ============================================================

missing_metrics = [
    metric for metric in metrics
    if metric not in test_df.columns
]

if missing_metrics:
    print("\nWARNING: These metric columns were not found:")
    print(missing_metrics)
    print("\nAvailable columns:")
    print(test_df.columns.tolist())
    raise ValueError(
        "Change the metric names in the 'metrics' list "
        "to match your CSV column names."
    )


# ============================================================
# 7. MEAN ± STANDARD DEVIATION
# ============================================================

summary_rows = []

for model in models:
    model_df = test_df[test_df["model"] == model]

    for metric in metrics:
        values = model_df[metric].dropna()

        mean = values.mean()
        sd = values.std(ddof=1)

        summary_rows.append({
            "Model": model,
            "Metric": metric,
            "Mean": mean,
            "SD": sd,
            "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",
            "N": len(values)
        })

summary_df = pd.DataFrame(summary_rows)

print("\n")
print("=" * 70)
print("MEAN ± STANDARD DEVIATION")
print("=" * 70)

summary_wide = summary_df.pivot(
    index="Model",
    columns="Metric",
    values="Mean ± SD"
)

print(summary_wide)


# ============================================================
# 8. CHECK MATCHED SEEDS
# ============================================================

print("\n")
print("=" * 70)
print("SEED CHECK")
print("=" * 70)

for model in models:
    model_seeds = sorted(
        test_df.loc[
            test_df["model"] == model,
            "seed"
        ].unique()
    )

    print(f"{model}: {model_seeds}")


# ============================================================
# 9. FRIEDMAN TEST
# ============================================================
#
# H0:
# There is no difference in performance among the six CNNs.
#
# H1:
# At least one CNN differs significantly.
#
# alpha = 0.05
# ============================================================

# friedman_results = []

# print("\n")
# print("=" * 70)
# print("FRIEDMAN TEST RESULTS")
# print("=" * 70)

# for metric in metrics:

#     pivot = test_df.pivot(
#         index="seed",
#         columns="model",
#         values=metric
#     )

#     # Force same model ordering
#     pivot = pivot[models]

#     # Remove any seed where one model is missing
#     pivot = pivot.dropna()

#     print(f"\nMetric: {metric}")
#     print("Number of matched runs:", len(pivot))

#     if len(pivot) < 2:
#         print("Not enough matched runs.")
#         continue

#     statistic, p_value = friedmanchisquare(
#         *[pivot[model].values for model in models]
#     )

#     significant = p_value < 0.05

#     friedman_results.append({
#         "Metric": metric,
#         "Friedman Chi-Square": statistic,
#         "p-value": p_value,
#         "Significant": significant
#     })

#     print(f"Friedman χ² = {statistic:.4f}")
#     print(f"p-value = {p_value:.6f}")

#     if significant:
#         print("Result: SIGNIFICANT (p < 0.05)")
#     else:
#         print("Result: NOT significant (p >= 0.05)")


# friedman_df = pd.DataFrame(friedman_results)

# ============================================================
# 9. FRIEDMAN TEST + KENDALL'S W EFFECT SIZE
# ============================================================
#
# H0:
# There is no difference in performance among the six CNNs.
#
# H1:
# At least one CNN differs significantly.
#
# Kendall's W is reported as an effect size for the Friedman test.
#
# W = chi-square / [n * (k - 1)]
#
# where:
# n = number of matched runs
# k = number of models
#
# Kendall's W ranges from 0 to 1:
# 0 = little/no agreement in model rankings across runs
# 1 = complete agreement in model rankings across runs
#
# alpha = 0.05
# ============================================================

friedman_results = []

print("\n")
print("=" * 70)
print("FRIEDMAN TEST + KENDALL'S W RESULTS")
print("=" * 70)

for metric in metrics:

    pivot = test_df.pivot(
        index="seed",
        columns="model",
        values=metric
    )

    # Force same model ordering
    pivot = pivot[models]

    # Remove any seed where one model is missing
    pivot = pivot.dropna()

    n = len(pivot)       # Number of matched runs
    k = len(models)      # Number of CNN architectures

    print(f"\nMetric: {metric}")
    print("Number of matched runs:", n)

    if n < 2:
        print("Not enough matched runs.")
        continue

    # Friedman test
    statistic, p_value = friedmanchisquare(
        *[pivot[model].values for model in models]
    )

    # Kendall's W effect size
    kendalls_w = statistic / (n * (k - 1))

    significant = p_value < 0.05

    friedman_results.append({
        "Metric": metric,
        "Friedman Chi-Square": statistic,
        "p-value": p_value,
        "Kendall's W": kendalls_w,
        "Significant": significant
    })

    print(f"Friedman χ² = {statistic:.4f}")
    print(f"p-value = {p_value:.6f}")
    print(f"Kendall's W = {kendalls_w:.4f}")

    if significant:
        print("Result: SIGNIFICANT (p < 0.05)")
    else:
        print("Result: NOT significant (p >= 0.05)")


friedman_df = pd.DataFrame(friedman_results)
# ============================================================
# 10. POST-HOC WILCOXON TESTS
# ============================================================
#
# Only run these when Friedman is significant.
#
# This compares ResNet-34, your best model, against each of
# the other five CNNs.
#
# Holm correction controls for multiple comparisons.
# ============================================================

reference_model = "ResNet-34"

posthoc_rows = []

print("\n")
print("=" * 70)
print("POST-HOC WILCOXON TESTS")
print("=" * 70)

for metric in metrics:

    friedman_row = friedman_df[
        friedman_df["Metric"] == metric
    ]

    if friedman_row.empty:
        continue

    friedman_p = friedman_row["p-value"].iloc[0]

    if friedman_p >= 0.05:
        print(
            f"\n{metric}: Friedman not significant; "
            "post-hoc test not performed."
        )
        continue

    pivot = test_df.pivot(
        index="seed",
        columns="model",
        values=metric
    )

    pivot = pivot[models].dropna()

    raw_pvalues = []
    comparison_names = []
    statistics = []

    for model in models:

        if model == reference_model:
            continue

        x = pivot[reference_model].values
        y = pivot[model].values

        try:
            stat, p = wilcoxon(
                x,
                y,
                alternative="two-sided",
                zero_method="wilcox"
            )

        except ValueError:
            # Happens if all paired differences are zero
            stat = 0
            p = 1.0

        comparison_names.append(
            f"{reference_model} vs {model}"
        )

        statistics.append(stat)
        raw_pvalues.append(p)

    # Holm multiple-comparison correction
    reject, adjusted_pvalues, _, _ = multipletests(
        raw_pvalues,
        alpha=0.05,
        method="holm"
    )

    print(f"\nMetric: {metric}")

    for (
        comparison,
        stat,
        raw_p,
        adjusted_p,
        sig
    ) in zip(
        comparison_names,
        statistics,
        raw_pvalues,
        adjusted_pvalues,
        reject
    ):

        posthoc_rows.append({
            "Metric": metric,
            "Comparison": comparison,
            "Wilcoxon statistic": stat,
            "Raw p-value": raw_p,
            "Holm-adjusted p-value": adjusted_p,
            "Significant": sig
        })

        print(
            f"{comparison}\n"
            f"  W = {stat:.4f}\n"
            f"  raw p = {raw_p:.6f}\n"
            f"  Holm-adjusted p = {adjusted_p:.6f}\n"
            f"  Significant = {'YES' if sig else 'NO'}"
        )


posthoc_df = pd.DataFrame(posthoc_rows)


# ============================================================
# 11. SAVE EVERYTHING
# ============================================================

combined_df.to_csv(
    "combined_all_cnn_multirun_results.csv",
    index=False
)

summary_df.to_csv(
    "cnn_mean_sd_results.csv",
    index=False
)

friedman_df.to_csv(
    "cnn_friedman_results.csv",
    index=False
)

posthoc_df.to_csv(
    "cnn_posthoc_wilcoxon_holm.csv",
    index=False
)

Combined shape: (60, 25)

Columns found:
['run_idx', 'seed', 'internal_normal.loss', 'internal_normal.accuracy', 'internal_normal.precision_macro', 'internal_normal.recall_macro', 'internal_normal.f1_macro', 'internal_normal.macro_roc_auc', 'internal_normal.weighted_roc_auc', 'internal_tta.loss', 'internal_tta.accuracy', 'internal_tta.precision_macro', 'internal_tta.recall_macro', 'internal_tta.f1_macro', 'internal_tta.macro_roc_auc', 'internal_tta.weighted_roc_auc', 'external_normal.accuracy', 'external_normal.precision', 'external_normal.recall', 'external_normal.macro_f1', 'external_tta.accuracy', 'external_tta.precision', 'external_tta.recall', 'external_tta.macro_f1', 'model']

Models:
model
ResNet-34          10
ResNet-50          10
DenseNet-121       10
EfficientNet-B0    10
Inception-V3       10
VGG-16             10
Name: count, dtype: int64

First few rows:
   run_idx  seed  internal_normal.loss  internal_normal.accuracy  \
0        0    42              0.818034             

In [3]:
import pandas as pd
import numpy as np

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. FILE PATHS
# ============================================================

files = {
    "ResNet-34": (
        "outputs/resnet34/"
        "resnet34_multirun_per_run.csv"
    ),
    "ResNet-50": (
        "outputs/resnet50/"
        "resnet50_multirun_per_run.csv"
    ),
    "DenseNet-121": (
        "outputs/densenet121/"
        "densenet121_multirun_per_run.csv"
    ),
    "EfficientNet-B0": (
        "outputs/efficientnetb0/"
        "efficientnetb0_multirun_per_run.csv"
    ),
    "Inception-V3": (
        "outputs/inceptionv3/"
        "inceptionv3_multirun_per_run.csv"
    ),
    "VGG-16": (
        "outputs/vgg16/"
        "vgg16_multirun_per_run.csv"
    ),
}


# ============================================================
# 2. DATASETS AND COLUMN NAMES
# ============================================================

evaluation_settings = {
    "Internal EBHI-SEG": {
        "normal_column": "internal_normal.accuracy",
        "tta_column": "internal_tta.accuracy"
    },
    "External LC25000": {
        "normal_column": "external_normal.accuracy",
        "tta_column": "external_tta.accuracy"
    }
}


# ============================================================
# 3. RUN WILCOXON TESTS
# ============================================================
#
# H0: TTA does not improve accuracy.
# H1: TTA improves accuracy.
#
# A one-sided Wilcoxon signed-rank test is used:
# alternative = "greater"
#
# Holm correction is applied separately for each dataset across
# the six architecture-level comparisons.
# ============================================================

all_results = []

for dataset_name, columns in evaluation_settings.items():

    dataset_results = []

    normal_column = columns["normal_column"]
    tta_column = columns["tta_column"]

    for model_name, file_path in files.items():

        df = pd.read_csv(file_path)

        required_columns = [
            "seed",
            normal_column,
            tta_column
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{model_name} is missing these columns: "
                f"{missing_columns}"
            )

        # Sort by seed so the same training runs are paired.
        paired_df = (
            df[
                [
                    "seed",
                    normal_column,
                    tta_column
                ]
            ]
            .dropna()
            .sort_values("seed")
            .copy()
        )

        normal = paired_df[normal_column].astype(float)
        tta = paired_df[tta_column].astype(float)

        improvement = tta - normal

        # If every paired difference is exactly zero, Wilcoxon
        # cannot be calculated meaningfully.
        if np.allclose(improvement, 0):
            statistic = 0.0
            raw_p_value = 1.0

        else:
            statistic, raw_p_value = wilcoxon(
                tta,
                normal,
                alternative="greater",
                zero_method="wilcox"
            )

        dataset_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Number of Runs": len(paired_df),

            "Normal Mean": normal.mean(),
            "Normal SD": normal.std(ddof=1),

            "TTA Mean": tta.mean(),
            "TTA SD": tta.std(ddof=1),

            "Mean Improvement": improvement.mean(),
            "SD Improvement": improvement.std(ddof=1),

            "Improved Runs": int((improvement > 0).sum()),
            "Unchanged Runs": int(
                np.isclose(improvement, 0).sum()
            ),
            "Worsened Runs": int((improvement < 0).sum()),

            "Wilcoxon W": statistic,
            "Raw p-value": raw_p_value
        })

    dataset_df = pd.DataFrame(dataset_results)

    # Holm correction across the six CNN tests for this dataset.
    reject, adjusted_p_values, _, _ = multipletests(
        dataset_df["Raw p-value"],
        alpha=0.05,
        method="holm"
    )

    dataset_df["Holm-adjusted p-value"] = (
        adjusted_p_values
    )

    dataset_df["Significant TTA Improvement"] = reject

    all_results.append(dataset_df)


# ============================================================
# 4. COMBINE INTERNAL AND EXTERNAL RESULTS
# ============================================================

results_df = pd.concat(
    all_results,
    ignore_index=True
)


# ============================================================
# 5. FORMAT MEAN ± SD
# ============================================================

results_df["Normal Accuracy"] = results_df.apply(
    lambda row: (
        f"{row['Normal Mean']:.4f} ± "
        f"{row['Normal SD']:.4f}"
    ),
    axis=1
)

results_df["TTA Accuracy"] = results_df.apply(
    lambda row: (
        f"{row['TTA Mean']:.4f} ± "
        f"{row['TTA SD']:.4f}"
    ),
    axis=1
)

results_df["Accuracy Improvement"] = results_df.apply(
    lambda row: (
        f"{row['Mean Improvement']:.4f} ± "
        f"{row['SD Improvement']:.4f}"
    ),
    axis=1
)


# Optional: improvement expressed as percentage points.
results_df["Mean Improvement (percentage points)"] = (
    results_df["Mean Improvement"] * 100
)


# ============================================================
# 6. FINAL TABLE
# ============================================================

final_table = results_df[
    [
        "Dataset",
        "Model",
        "Normal Accuracy",
        "TTA Accuracy",
        "Accuracy Improvement",
        "Mean Improvement (percentage points)",
        "Improved Runs",
        "Unchanged Runs",
        "Worsened Runs",
        "Wilcoxon W",
        "Raw p-value",
        "Holm-adjusted p-value",
        "Significant TTA Improvement"
    ]
].copy()


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

for dataset_name in evaluation_settings:

    print("\n")
    print("=" * 100)
    print(dataset_name.upper())
    print("=" * 100)

    display_table = final_table[
        final_table["Dataset"] == dataset_name
    ]

    print(display_table.to_string(index=False))



INTERNAL EBHI-SEG
          Dataset           Model Normal Accuracy    TTA Accuracy Accuracy Improvement  Mean Improvement (percentage points)  Improved Runs  Unchanged Runs  Worsened Runs  Wilcoxon W  Raw p-value  Holm-adjusted p-value  Significant TTA Improvement
Internal EBHI-SEG       ResNet-34 0.8952 ± 0.0162 0.9101 ± 0.0094      0.0149 ± 0.0124                              1.492537              9               0              1        51.0     0.006836               0.013672                         True
Internal EBHI-SEG       ResNet-50 0.8910 ± 0.0095 0.9036 ± 0.0115      0.0125 ± 0.0097                              1.253731              9               1              0        45.0     0.001953               0.009766                         True
Internal EBHI-SEG    DenseNet-121 0.8910 ± 0.0162 0.9057 ± 0.0101      0.0146 ± 0.0099                              1.462687              9               0              1        54.0     0.001953               0.009766                  

In [4]:
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. FILE PATHS
# ============================================================

files = {
    "ResNet-34":
        "outputs/resnet34/resnet34_multirun_per_run.csv",

    "ResNet-50":
        "outputs/resnet50/resnet50_multirun_per_run.csv",

    "DenseNet-121":
        "outputs/densenet121/densenet121_multirun_per_run.csv",

    "EfficientNet-B0":
        "outputs/efficientnetb0/"
        "efficientnetb0_multirun_per_run.csv",

    "Inception-V3":
        "outputs/inceptionv3/"
        "inceptionv3_multirun_per_run.csv",

    "VGG-16":
        "outputs/vgg16/vgg16_multirun_per_run.csv",
}

models = list(files.keys())
reference_model = "ResNet-34"
expected_runs = 10
alpha = 0.05


# ============================================================
# 2. LOAD AND COMBINE RESULTS
# ============================================================

all_dfs = []

for model_name, file_path in files.items():
    df = pd.read_csv(file_path)
    df["model"] = model_name
    all_dfs.append(df)

combined_df = pd.concat(
    all_dfs,
    ignore_index=True
)

print("Combined shape:", combined_df.shape)

print("\nColumns found:")
for column in combined_df.columns:
    print(column)


# ============================================================
# 3. EXTERNAL METRICS WITH TTA
# ============================================================

external_metrics = [
    "external_tta.accuracy",
    "external_tta.macro_f1"
]

missing_metrics = [
    metric
    for metric in external_metrics
    if metric not in combined_df.columns
]

if missing_metrics:
    print("\nMissing external metric columns:")
    print(missing_metrics)

    print("\nAvailable external columns:")
    print([
        column
        for column in combined_df.columns
        if "external" in column.lower()
    ])

    raise ValueError(
        "Change the external metric names to match "
        "the available CSV columns."
    )


# ============================================================
# 4. CHECK MODEL-SEED STRUCTURE
# ============================================================

if "seed" not in combined_df.columns:
    raise ValueError(
        "The combined data do not contain a 'seed' column."
    )

duplicate_rows = combined_df.duplicated(
    subset=["model", "seed"],
    keep=False
)

if duplicate_rows.any():
    print("\nDuplicate model-seed combinations:")
    print(
        combined_df.loc[
            duplicate_rows,
            ["model", "seed"]
        ].sort_values(["model", "seed"])
    )

    raise ValueError(
        "Each model must have only one row for each seed."
    )

seed_sets = {}

for model in models:
    model_seeds = set(
        combined_df.loc[
            combined_df["model"] == model,
            "seed"
        ].tolist()
    )

    seed_sets[model] = model_seeds

reference_seeds = seed_sets[models[0]]

for model in models:
    if seed_sets[model] != reference_seeds:
        raise ValueError(
            f"Seed mismatch detected for {model}."
        )

if len(reference_seeds) != expected_runs:
    raise ValueError(
        f"Expected {expected_runs} matched seeds, "
        f"but found {len(reference_seeds)}."
    )

print("\nAll models use the same matched seeds:")
print(sorted(reference_seeds))


# ============================================================
# 5. EXTERNAL MEAN ± STANDARD DEVIATION
# ============================================================

summary_rows = []

for model in models:
    model_df = combined_df[
        combined_df["model"] == model
    ]

    for metric in external_metrics:
        values = model_df[metric].dropna()

        summary_rows.append({
            "Dataset": "External LC25000",
            "Setting": "TTA",
            "Model": model,
            "Metric": metric,
            "Mean": values.mean(),
            "SD": values.std(ddof=1),
            "Mean ± SD":
                f"{values.mean():.4f} ± "
                f"{values.std(ddof=1):.4f}",
            "N": len(values)
        })

external_summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 85)
print("EXTERNAL LC25000 MEAN ± SD WITH TTA")
print("=" * 85)

summary_wide = external_summary_df.pivot(
    index="Model",
    columns="Metric",
    values="Mean ± SD"
)

print(summary_wide.to_string())


# ============================================================
# 6. EXTERNAL FRIEDMAN TESTS
# ============================================================

friedman_rows = []
posthoc_rows = []

print("\n" + "=" * 85)
print("EXTERNAL LC25000 FRIEDMAN TESTS")
print("=" * 85)

for metric in external_metrics:

    pivot = combined_df.pivot(
        index="seed",
        columns="model",
        values=metric
    )

    pivot = pivot[models].dropna()

    if len(pivot) != expected_runs:
        raise ValueError(
            f"{metric}: expected {expected_runs} "
            f"matched runs, but found {len(pivot)}."
        )

    statistic, p_value = friedmanchisquare(
        *[
            pivot[model].to_numpy()
            for model in models
        ]
    )

    degrees_of_freedom = len(models) - 1

    # Friedman effect size
    kendalls_w = statistic / (
        len(pivot) * degrees_of_freedom
    )

    significant = p_value < alpha

    friedman_rows.append({
        "Dataset": "External LC25000",
        "Setting": "TTA",
        "Metric": metric,
        "Number of matched runs": len(pivot),
        "Friedman Chi-Square": statistic,
        "Degrees of Freedom": degrees_of_freedom,
        "p-value": p_value,
        "Kendall's W": kendalls_w,
        "Significant": significant
    })

    print(f"\nMetric: {metric}")
    print(f"Matched runs: {len(pivot)}")
    print(
        f"Friedman χ²({degrees_of_freedom}) "
        f"= {statistic:.4f}"
    )
    print(f"p-value = {p_value:.6f}")
    print(f"Kendall's W = {kendalls_w:.4f}")
    print(
        "Result:",
        "SIGNIFICANT" if significant
        else "NOT SIGNIFICANT"
    )

    # ========================================================
    # 7. POST-HOC WILCOXON TESTS
    # ========================================================

    if not significant:
        print(
            "Post-hoc comparisons were not performed."
        )
        continue

    raw_pvalues = []
    comparison_information = []

    for comparison_model in models:

        if comparison_model == reference_model:
            continue

        reference_values = (
            pivot[reference_model].to_numpy()
        )

        comparison_values = (
            pivot[comparison_model].to_numpy()
        )

        differences = (
            reference_values - comparison_values
        )

        try:
            wilcoxon_w, raw_p = wilcoxon(
                reference_values,
                comparison_values,
                alternative="two-sided",
                zero_method="wilcox"
            )

        except ValueError:
            # Occurs when every paired difference is zero.
            wilcoxon_w = 0.0
            raw_p = 1.0

        raw_pvalues.append(raw_p)

        comparison_information.append({
            "Comparison":
                f"{reference_model} vs "
                f"{comparison_model}",

            "Comparison Model":
                comparison_model,

            "Reference Mean":
                reference_values.mean(),

            "Comparison Mean":
                comparison_values.mean(),

            "Mean Difference":
                differences.mean(),

            "Wilcoxon W":
                wilcoxon_w,

            "Raw p-value":
                raw_p
        })

    reject, adjusted_pvalues, _, _ = multipletests(
        raw_pvalues,
        alpha=alpha,
        method="holm"
    )

    print("\nPost-hoc comparisons:")

    for information, adjusted_p, rejected in zip(
        comparison_information,
        adjusted_pvalues,
        reject
    ):

        row = {
            "Dataset": "External LC25000",
            "Setting": "TTA",
            "Metric": metric,
            **information,
            "Holm-adjusted p-value":
                adjusted_p,
            "Significant":
                bool(rejected)
        }

        posthoc_rows.append(row)

        print(
            f"\n{information['Comparison']}"
            f"\n  Reference mean = "
            f"{information['Reference Mean']:.4f}"
            f"\n  Comparison mean = "
            f"{information['Comparison Mean']:.4f}"
            f"\n  Mean difference = "
            f"{information['Mean Difference']:.4f}"
            f"\n  W = "
            f"{information['Wilcoxon W']:.4f}"
            f"\n  Raw p = "
            f"{information['Raw p-value']:.6f}"
            f"\n  Holm-adjusted p = "
            f"{adjusted_p:.6f}"
            f"\n  Significant = "
            f"{'YES' if rejected else 'NO'}"
        )

Combined shape: (60, 25)

Columns found:
run_idx
seed
internal_normal.loss
internal_normal.accuracy
internal_normal.precision_macro
internal_normal.recall_macro
internal_normal.f1_macro
internal_normal.macro_roc_auc
internal_normal.weighted_roc_auc
internal_tta.loss
internal_tta.accuracy
internal_tta.precision_macro
internal_tta.recall_macro
internal_tta.f1_macro
internal_tta.macro_roc_auc
internal_tta.weighted_roc_auc
external_normal.accuracy
external_normal.precision
external_normal.recall
external_normal.macro_f1
external_tta.accuracy
external_tta.precision
external_tta.recall
external_tta.macro_f1
model

All models use the same matched seeds:
[7, 42, 77, 123, 314, 2024, 2025, 3407, 8888, 9999]

EXTERNAL LC25000 MEAN ± SD WITH TTA
Metric          external_tta.accuracy external_tta.macro_f1
Model                                                      
DenseNet-121          0.5411 ± 0.0678       0.6081 ± 0.0780
EfficientNet-B0       0.4375 ± 0.0502       0.5474 ± 0.0520
Inception-V3    

In [5]:
import pandas as pd
import numpy as np

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1. FILE PATHS
# ============================================================

files = {
    "ResNet-34": (
        "outputs/resnet34/"
        "resnet34_multirun_per_run.csv"
    ),
    "ResNet-50": (
        "outputs/resnet50/"
        "resnet50_multirun_per_run.csv"
    ),
    "DenseNet-121": (
        "outputs/densenet121/"
        "densenet121_multirun_per_run.csv"
    ),
    "EfficientNet-B0": (
        "outputs/efficientnetb0/"
        "efficientnetb0_multirun_per_run.csv"
    ),
    "Inception-V3": (
        "outputs/inceptionv3/"
        "inceptionv3_multirun_per_run.csv"
    ),
    "VGG-16": (
        "outputs/vgg16/"
        "vgg16_multirun_per_run.csv"
    ),
}


# ============================================================
# 2. DATASETS AND COLUMN NAMES
# ============================================================

evaluation_settings = {
    "Internal EBHI-SEG": {
        "normal_column": "internal_normal.accuracy",
        "tta_column": "internal_tta.accuracy"
    },
    "External LC25000": {
        "normal_column": "external_normal.accuracy",
        "tta_column": "external_tta.accuracy"
    }
}


# ============================================================
# 3. RUN WILCOXON TESTS
# ============================================================
#
# H0: TTA does not improve accuracy.
# H1: TTA improves accuracy.
#
# A one-sided Wilcoxon signed-rank test is used:
# alternative = "greater"
#
# Holm correction is applied separately for each dataset across
# the six architecture-level comparisons.
# ============================================================

all_results = []

for dataset_name, columns in evaluation_settings.items():

    dataset_results = []

    normal_column = columns["normal_column"]
    tta_column = columns["tta_column"]

    for model_name, file_path in files.items():

        df = pd.read_csv(file_path)

        required_columns = [
            "seed",
            normal_column,
            tta_column
        ]

        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{model_name} is missing these columns: "
                f"{missing_columns}"
            )

        # Sort by seed so the same training runs are paired.
        paired_df = (
            df[
                [
                    "seed",
                    normal_column,
                    tta_column
                ]
            ]
            .dropna()
            .sort_values("seed")
            .copy()
        )

        normal = paired_df[normal_column].astype(float)
        tta = paired_df[tta_column].astype(float)

        improvement = tta - normal

        # If every paired difference is exactly zero, Wilcoxon
        # cannot be calculated meaningfully.
        if np.allclose(improvement, 0):
            statistic = 0.0
            raw_p_value = 1.0

        else:
            statistic, raw_p_value = wilcoxon(
                tta,
                normal,
                alternative="greater",
                zero_method="wilcox"
            )

        dataset_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Number of Runs": len(paired_df),

            "Normal Mean": normal.mean(),
            "Normal SD": normal.std(ddof=1),

            "TTA Mean": tta.mean(),
            "TTA SD": tta.std(ddof=1),

            "Mean Improvement": improvement.mean(),
            "SD Improvement": improvement.std(ddof=1),

            "Improved Runs": int((improvement > 0).sum()),
            "Unchanged Runs": int(
                np.isclose(improvement, 0).sum()
            ),
            "Worsened Runs": int((improvement < 0).sum()),

            "Wilcoxon W": statistic,
            "Raw p-value": raw_p_value
        })

    dataset_df = pd.DataFrame(dataset_results)

    # Holm correction across the six CNN tests for this dataset.
    reject, adjusted_p_values, _, _ = multipletests(
        dataset_df["Raw p-value"],
        alpha=0.05,
        method="holm"
    )

    dataset_df["Holm-adjusted p-value"] = (
        adjusted_p_values
    )

    dataset_df["Significant TTA Improvement"] = reject

    all_results.append(dataset_df)


# ============================================================
# 4. COMBINE INTERNAL AND EXTERNAL RESULTS
# ============================================================

results_df = pd.concat(
    all_results,
    ignore_index=True
)


# ============================================================
# 5. FORMAT MEAN ± SD
# ============================================================

results_df["Normal Accuracy"] = results_df.apply(
    lambda row: (
        f"{row['Normal Mean']:.4f} ± "
        f"{row['Normal SD']:.4f}"
    ),
    axis=1
)

results_df["TTA Accuracy"] = results_df.apply(
    lambda row: (
        f"{row['TTA Mean']:.4f} ± "
        f"{row['TTA SD']:.4f}"
    ),
    axis=1
)

results_df["Accuracy Improvement"] = results_df.apply(
    lambda row: (
        f"{row['Mean Improvement']:.4f} ± "
        f"{row['SD Improvement']:.4f}"
    ),
    axis=1
)


# Optional: improvement expressed as percentage points.
results_df["Mean Improvement (percentage points)"] = (
    results_df["Mean Improvement"] * 100
)


# ============================================================
# 6. FINAL TABLE
# ============================================================

final_table = results_df[
    [
        "Dataset",
        "Model",
        "Normal Accuracy",
        "TTA Accuracy",
        "Accuracy Improvement",
        "Mean Improvement (percentage points)",
        "Improved Runs",
        "Unchanged Runs",
        "Worsened Runs",
        "Wilcoxon W",
        "Raw p-value",
        "Holm-adjusted p-value",
        "Significant TTA Improvement"
    ]
].copy()


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

for dataset_name in evaluation_settings:

    print("\n")
    print("=" * 100)
    print(dataset_name.upper())
    print("=" * 100)

    display_table = final_table[
        final_table["Dataset"] == dataset_name
    ]

    print(display_table.to_string(index=False))


# ============================================================
# 8. SAVE RESULTS
# ============================================================

final_table.to_csv(
    "tta_internal_external_wilcoxon_holm_results.csv",
    index=False
)

internal_table = final_table[
    final_table["Dataset"] == "Internal EBHI-SEG"
]

external_table = final_table[
    final_table["Dataset"] == "External LC25000"
]

internal_table.to_csv(
    "tta_internal_ebhiseg_wilcoxon_holm_results.csv",
    index=False
)

external_table.to_csv(
    "tta_external_lc25000_wilcoxon_holm_results.csv",
    index=False
)

print("\nFiles saved:")
print(
    "tta_internal_external_wilcoxon_holm_results.csv"
)
print(
    "tta_internal_ebhiseg_wilcoxon_holm_results.csv"
)
print(
    "tta_external_lc25000_wilcoxon_holm_results.csv"
)



INTERNAL EBHI-SEG
          Dataset           Model Normal Accuracy    TTA Accuracy Accuracy Improvement  Mean Improvement (percentage points)  Improved Runs  Unchanged Runs  Worsened Runs  Wilcoxon W  Raw p-value  Holm-adjusted p-value  Significant TTA Improvement
Internal EBHI-SEG       ResNet-34 0.8952 ± 0.0162 0.9101 ± 0.0094      0.0149 ± 0.0124                              1.492537              9               0              1        51.0     0.006836               0.013672                         True
Internal EBHI-SEG       ResNet-50 0.8910 ± 0.0095 0.9036 ± 0.0115      0.0125 ± 0.0097                              1.253731              9               1              0        45.0     0.001953               0.009766                         True
Internal EBHI-SEG    DenseNet-121 0.8910 ± 0.0162 0.9057 ± 0.0101      0.0146 ± 0.0099                              1.462687              9               0              1        54.0     0.001953               0.009766                  

In [6]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import ttest_rel


metrics = {
    "Accuracy": "internal_tta.accuracy",
    "Macro Precision": "internal_tta.precision_macro",
    "Macro Recall": "internal_tta.recall_macro",
    "Macro F1": "internal_tta.f1_macro",
    "Macro ROC-AUC": "internal_tta.macro_roc_auc"
}

models = [
    "ResNet-34",
    "ResNet-50",
    "DenseNet-121",
    "EfficientNet-B0",
    "Inception-V3",
    "VGG-16"
]

results = []

for metric_name, column in metrics.items():

    pivot = combined_df.pivot(
        index="seed",
        columns="model",
        values=column
    )

    for model1, model2 in combinations(models, 2):

        x = pivot[model1].values
        y = pivot[model2].values

        # paired differences
        differences = x - y

        mean_diff = np.mean(differences)
        sd_diff = np.std(differences, ddof=1)

        # Cohen's dz for paired samples
        if sd_diff != 0:
            cohens_dz = mean_diff / sd_diff
        else:
            cohens_dz = np.inf

        # Paired t-test just for additional information
        t_stat, p = ttest_rel(x, y)

        results.append({
            "Metric": metric_name,
            "Model 1": model1,
            "Model 2": model2,
            "Mean Difference": mean_diff,
            "SD Difference": sd_diff,
            "Cohen's dz": cohens_dz,
            "|Cohen's dz|": abs(cohens_dz),
            "Paired t p-value": p
        })


effect_df = pd.DataFrame(results)

# Sort within each metric by effect size
effect_df = effect_df.sort_values(
    ["Metric", "|Cohen's dz|"],
    ascending=[True, False]
)

print(effect_df.to_string(index=False))

effect_df.to_csv(
    "pairwise_effect_sizes_5runs.csv",
    index=False
)

         Metric         Model 1         Model 2  Mean Difference  SD Difference  Cohen's dz  |Cohen's dz|  Paired t p-value
       Accuracy       ResNet-50    Inception-V3         0.040896       0.009339    4.378787      4.378787      2.255085e-07
       Accuracy       ResNet-34    Inception-V3         0.047463       0.013008    3.648777      3.648777      1.074667e-06
       Accuracy       ResNet-50 EfficientNet-B0         0.043284       0.013948    3.103187      3.103187      4.185615e-06
       Accuracy       ResNet-34 EfficientNet-B0         0.049851       0.018350    2.716655      2.716655      1.247271e-05
       Accuracy    DenseNet-121 EfficientNet-B0         0.045373       0.018119    2.504134      2.504134      2.401307e-05
       Accuracy    DenseNet-121    Inception-V3         0.042985       0.017246    2.492492      2.492492      2.492133e-05
       Accuracy    Inception-V3          VGG-16        -0.036119       0.017375   -2.078873      2.078873      1.022980e-04
       A